# Video 09 - Keeping agents safe

**You are here:** Video 9 of 13 - the Agent phase finishes (see `../you-are-here.svg`). On the agent anatomy picture, today lights up **Guardrails**.

**The one idea:** a working loop is not enough. Wrap it in **seatbelts** so it cannot run away or crash your notebook.

**By the end you can:**
1. Turn tool failures into results the model can read.
2. Stop on truncation, repeats, and budgets.
3. Gate write tools behind approval.

**Before you start:** attach this notebook to **Serverless**.


In [ ]:
import json, time
from datetime import date
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"

ORDERS = {42: {"customer": "Alice", "total": 250.0}, 7: {"customer": "Bob", "total": 80.0}}

def get_order_total(customer_id: int):
    order = ORDERS.get(int(customer_id))
    return order["total"] if order else "customer not found"

def calculator(expression: str):
    if not set(expression) <= set("0123456789+-*/(). "):
        return "error: only basic arithmetic is allowed"
    return eval(expression, {"__builtins__": {}})

def get_today():
    return date.today().isoformat()

def tool_spec(name, description, properties=None, required=None):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties or {}, "required": required or []}}}

TOOL_SPECS = [
    tool_spec("get_order_total", "Look up the order total in dollars for a customer id.",
              {"customer_id": {"type": "integer"}}, ["customer_id"]),
    tool_spec("calculator", "Evaluate a basic arithmetic expression, e.g. '250 * 0.15'.",
              {"expression": {"type": "string"}}, ["expression"]),
    tool_spec("get_today", "Get today's date in ISO format."),
]
TOOLS = {"get_order_total": get_order_total, "calculator": calculator, "get_today": get_today}
WRITE_TOOLS = set()  # filled later when we demo cancel_order

print("Connected.")


## Video 08 agent (baseline)

Happy path still works. Guardrails matter on the *unhappy* paths.


In [ ]:
def add_tool_results(messages, msg, functions):
    messages.append(msg.model_dump(exclude_none=True))
    for call in msg.tool_calls:
        args = json.loads(call.function.arguments or "{}")
        func = functions.get(call.function.name)
        result = func(**args) if func else f"error: unknown tool {call.function.name}"
        content = result if isinstance(result, str) else json.dumps(result)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": content})

def run_agent(question, max_steps=5, verbose=True):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the tools when you need facts or maths."},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SPECS)
        msg = response.choices[0].message
        if not msg.tool_calls:
            return msg.content
        if verbose:
            for tc in msg.tool_calls:
                print(f"  step {step}: {tc.function.name}({tc.function.arguments})")
        add_tool_results(messages, msg, TOOLS)
    return "Stopped: reached max_steps without a final answer."

print(run_agent("What's 15% of the order total for customer 42?"))


## Errors as results, not crashes

If a tool raises, the loop dies - unless you catch it and send the error back as the tool content.


In [ ]:
def get_order_total_strict(customer_id: int):
    order = ORDERS.get(int(customer_id))
    if not order:
        raise ValueError("unknown customer")
    return order["total"]

# Without a try/except this kills the cell:
try:
    broken = {"get_order_total": get_order_total_strict, "calculator": calculator, "get_today": get_today}
    print(run_agent("What is the order total for customer 999?", verbose=True))
    # force the strict tool via a one-off call path
    messages = [{"role": "user", "content": "What is the order total for customer 999?"}]
    msg = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SPECS).choices[0].message
    messages.append(msg.model_dump(exclude_none=True))
    call = msg.tool_calls[0]
    args = json.loads(call.function.arguments or "{}")
    print("About to call strict tool with", args)
    result = get_order_total_strict(**args)  # boom
except Exception as e:
    print("CRASH:", type(e).__name__, "-", e)


In [ ]:
def run_tool_safe(name, args, functions):
    """Always return a string result - never raise into the loop."""
    func = functions.get(name)
    if func is None:
        return f"error: unknown tool {name}"
    try:
        result = func(**args)
        return result if isinstance(result, str) else json.dumps(result)
    except Exception as e:
        return f"error: {e}"

def run_agent_safe_tools(question, max_steps=5, functions=None, verbose=True):
    functions = functions or TOOLS
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the tools when you need facts or maths."},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SPECS)
        msg = response.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg.model_dump(exclude_none=True))
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            content = run_tool_safe(call.function.name, args, functions)
            if verbose:
                print(f"  step {step}: {call.function.name}({args}) -> {content}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": content})
    return "Stopped: reached max_steps."

strict = {"get_order_total": get_order_total_strict, "calculator": calculator, "get_today": get_today}
print(run_agent_safe_tools("What is the order total for customer 999?", functions=strict))


## Don't trust a truncated reply

`finish_reason == "length"` means the text was cut off. No tool calls does **not** mean "done".


In [ ]:
messages = [
    {"role": "system", "content": "Write a long, detailed essay about why agent loops need guardrails."},
    {"role": "user", "content": "Explain in at least 20 sentences."},
]
response = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=16)
choice = response.choices[0]
print("finish_reason:", choice.finish_reason)
print("content:", repr(choice.message.content))
print("tool_calls:", choice.message.tool_calls)
if choice.finish_reason == "length":
    print("-> Truncated. Do NOT return this as the final answer.")


## Stop repeated tool calls

If the model asks for the same tool with the same arguments again and again, stop.


In [ ]:
def run_agent_repeat_guard(question, max_steps=5, max_same=2, verbose=True):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the tools when you need facts or maths."},
        {"role": "user", "content": question},
    ]
    seen = {}
    n_calls = 0
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SPECS)
        n_calls += 1
        msg = response.choices[0].message
        if not msg.tool_calls:
            return msg.content, n_calls
        messages.append(msg.model_dump(exclude_none=True))
        for call in msg.tool_calls:
            fp = f"{call.function.name}:{call.function.arguments}"
            seen[fp] = seen.get(fp, 0) + 1
            if seen[fp] > max_same:
                return f"Stopped: repeated the same tool call ({fp})", n_calls
            args = json.loads(call.function.arguments or "{}")
            content = run_tool_safe(call.function.name, args, TOOLS)
            if verbose:
                print(f"  step {step}: {fp} (seen={seen[fp]}) -> {content}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": content})
    return "Stopped: max_steps", n_calls

# Demo the idea with a fingerprint counter on a normal question (usually no repeats)
answer, n = run_agent_repeat_guard("What's 15% of the order total for customer 42?")
print("Answer:", answer)
print("Model calls:", n)


## Allow-list + read vs write

Only run tools you defined. Treat write tools as gated.


In [ ]:
def cancel_order(order_id: int):
    return f"cancelled order {order_id}"  # would change the world in production

WRITE_TOOLS = {"cancel_order"}
ALL_TOOLS = {**TOOLS, "cancel_order": cancel_order}
ALL_SPECS = TOOL_SPECS + [
    tool_spec("cancel_order", "Cancel an order by id. This changes data - use only with approval.",
              {"order_id": {"type": "integer"}}, ["order_id"]),
]

def run_tool_gated(name, args, functions, allow_writes=False):
    if name not in functions:
        return f"error: unknown tool {name}"
    if name in WRITE_TOOLS and not allow_writes:
        return "error: write requires approval"
    try:
        result = functions[name](**args)
        return result if isinstance(result, str) else json.dumps(result)
    except Exception as e:
        return f"error: {e}"

# Simulate the model asking to cancel - we refuse without allow_writes
print("Refused:", run_tool_gated("cancel_order", {"order_id": 42}, ALL_TOOLS, allow_writes=False))
print("Allowed:", run_tool_gated("cancel_order", {"order_id": 42}, ALL_TOOLS, allow_writes=True))
print("Unknown:", run_tool_gated("delete_database", {}, ALL_TOOLS))


## Token budget

Sum tokens each step. Cross the budget → stop cleanly.


In [ ]:
def run_agent_budget(question, max_steps=5, max_tokens_total=2000, verbose=True):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the tools when you need facts or maths."},
        {"role": "user", "content": question},
    ]
    tokens_used = 0
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SPECS)
        tokens_used += response.usage.total_tokens
        if verbose:
            print(f"  step {step}: +{response.usage.total_tokens} tokens (total {tokens_used})")
        if tokens_used > max_tokens_total:
            return f"Stopped: token budget exceeded ({tokens_used} > {max_tokens_total})."
        msg = response.choices[0].message
        if response.choices[0].finish_reason == "length":
            return "Stopped: reply was truncated (finish_reason=length)."
        if not msg.tool_calls:
            return msg.content
        messages.append(msg.model_dump(exclude_none=True))
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            content = run_tool_safe(call.function.name, args, TOOLS)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": content})
    return "Stopped: max_steps."

print(run_agent_budget("What's 15% of the order total for customer 42?", max_tokens_total=50000))
print("--- tight budget ---")
print(run_agent_budget("What's 15% of the order total for customer 42?", max_tokens_total=50))


In [ ]:
PRICE_PER_1M = 0.50  # EXAMPLE USD - replace with your real rate
answer, n_calls = run_agent_repeat_guard("How much more did customer 42 spend than customer 7?", max_steps=6)
# recount tokens properly
messages = [
    {"role": "system", "content": "You are a helpful assistant. Use the tools when you need facts or maths."},
    {"role": "user", "content": "How much more did customer 42 spend than customer 7?"},
]
tokens = 0
calls = 0
for step in range(1, 6):
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SPECS)
    calls += 1
    tokens += response.usage.total_tokens
    msg = response.choices[0].message
    if not msg.tool_calls:
        break
    messages.append(msg.model_dump(exclude_none=True))
    for call in msg.tool_calls:
        args = json.loads(call.function.arguments or "{}")
        content = run_tool_safe(call.function.name, args, TOOLS)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": content})

print(f"Model calls: {calls}")
print(f"Total tokens: {tokens}")
print(f"Example cost: ${tokens / 1_000_000 * PRICE_PER_1M:.4f}")
print("-> Guardrails protect correctness and the bill.")


## Safe agent: all seatbelts

One loop with: step limit, truncation check, errors as results, allow-list, write gate, repeat detector, token budget.


In [ ]:
def run_safe_agent(
    question,
    max_steps=5,
    max_tokens_total=50_000,
    max_same=2,
    allow_writes=False,
    functions=None,
    specs=None,
    verbose=True,
):
    functions = functions or TOOLS
    specs = specs or TOOL_SPECS
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the tools when you need facts or maths. Prefer read tools. Never invent tools."},
        {"role": "user", "content": question},
    ]
    seen, tokens_used = {}, 0
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=specs)
        tokens_used += response.usage.total_tokens
        if tokens_used > max_tokens_total:
            return f"Stopped: token budget exceeded ({tokens_used} > {max_tokens_total})."
        choice = response.choices[0]
        if choice.finish_reason == "length":
            return "Stopped: reply was truncated (finish_reason=length)."
        msg = choice.message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg.model_dump(exclude_none=True))
        for call in msg.tool_calls:
            fp = f"{call.function.name}:{call.function.arguments}"
            seen[fp] = seen.get(fp, 0) + 1
            if seen[fp] > max_same:
                return f"Stopped: repeated the same tool call ({fp})."
            args = json.loads(call.function.arguments or "{}")
            content = run_tool_gated(call.function.name, args, functions, allow_writes=allow_writes)
            if verbose:
                print(f"  step {step}: {call.function.name}({args}) -> {content}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": content})
    return "Stopped: reached max_steps without a final answer."

for q in [
    "What's 15% of the order total for customer 42?",
    "What is the order total for customer 999?",
    "Tell me a fun fact about Python.",
]:
    print("Q:", q)
    print("A:", run_safe_agent(q), "\n")


## Recap

* **Guardrails** = caps + checks around the Video 08 loop.
* **Errors as results** keep the notebook alive.
* Stop on **truncation**, **repeats**, **budgets**, and **disallowed writes**.
* Prompting helps; **code** seatbelts are what you trust.

**Exercises:** see `../exercises.md`. **Quiz:** `../quiz.md`. **Cheatsheet:** `../cheatsheet.md`.

**Next:** Video 10 - Seeing inside the agent (MLflow tracing).
